<a href="https://colab.research.google.com/github/alissonmenezes1990/Dio-Assistente-de-Voz-gerador-de-imagem/blob/main/Assistente_de_Voz_Multi_Idiomas_Com_Whisper_e_ChatGPT_Gerador_de_Imagens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
language = 'pt'


# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [ ]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [ ]:
# Instala as dependencias do Pip para o Whisper e para a geração da imagem
!pip install git+https://github.com/openai/whisper.git diffusers transformers accelerate torch -q

In [ ]:
import whisper
import torch

# Selecione o modelo do Whisper que melhor atenda  s suas necessidades:
model = whisper.load_model("small")

# Transcreve o audio gravado anteriormente.
# Ensure record_file exists from the previous recording cell
transcription = ""
try:
    result = model.transcribe(record_file, fp16=False, language=language)
    transcription = result["text"]
    print(f"Transcrição: {transcription}")
except NameError:
    print("Erro: Certifique-se de executar a c lula de grava  o primeiro.")

# 3. Geração de imagem

In [ ]:

from diffusers import StableDiffusionPipeline
import torch
from PIL import Image

# Define o dispositivo (usar GPU se disponível, senão CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Carrega o modelo
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch_dtype
).to(device)

# Usar a variável 'transcription' que vem do áudio
if 'transcription' in globals() and transcription:
    print(f"Gerando imagem para: {transcription}")
    # Gera a imagem
    image = pipe(transcription).images[0]
    # Mostra no Colab
    display(image)
else:
    print("Erro: Nenhuma transcrição encontrada. Grave um áudio primeiro.")
